In [1]:
from datetime import datetime
import os
import time

import numpy as np
import pandas as pd


In [2]:

from sherlock.functional import extract_features_to_csv
from sherlock.features.paragraph_vectors import initialise_pretrained_model, initialise_nltk
from sherlock.features.preprocessing import (
    #extract_features,
    #convert_string_lists_to_lists,
    prepare_feature_extraction,
    load_parquet_values,
)
from sherlock.features.word_embeddings import initialise_word_embeddings

unable to import 'smart_open.gcs', disabling that module


In [3]:
print(f'Started at {datetime.now()}.')

Started at 2025-05-12 13:30:30.757078.


In [4]:
prepare_feature_extraction()

Preparing feature extraction by downloading 4 files:
        
 ../sherlock/features/glove.6B.50d.txt, 
 ../sherlock/features/par_vec_trained_400.pkl.docvecs.vectors_docs.npy,
        
 ../sherlock/features/par_vec_trained_400.pkl.trainables.syn1neg.npy, and 
 ../sherlock/features/par_vec_trained_400.pkl.wv.vectors.npy.
        
All files for extracting word and paragraph embeddings are present.


In [5]:
if not os.path.exists('../sherlock/features/par_vec_trained_400.pkl.docvecs.vectors_docs.npy'):
    raise SystemExit(
        """
        Trained paragraph vectors do not exist,
        please run the '01-train-paragraph-vector-features' notebook before continuing
        """
    )

In [6]:
# ensure embedding initialisation is outside of timing for extract_features
prepare_feature_extraction()
initialise_word_embeddings()
initialise_pretrained_model(400)
initialise_nltk()

Preparing feature extraction by downloading 4 files:
        
 ../sherlock/features/glove.6B.50d.txt, 
 ../sherlock/features/par_vec_trained_400.pkl.docvecs.vectors_docs.npy,
        
 ../sherlock/features/par_vec_trained_400.pkl.trainables.syn1neg.npy, and 
 ../sherlock/features/par_vec_trained_400.pkl.wv.vectors.npy.
        
All files for extracting word and paragraph embeddings are present.
Initialising word embeddings
Initialise Word Embeddings process took 0:00:02.705999 seconds.
Initialise Doc2Vec Model, 400 dim, process took 0:00:01.297964 seconds. (filename = ../sherlock/features/par_vec_trained_400.pkl)
Initialised NLTK, process took 0:00:00.166111 seconds.


[nltk_data] Downloading package punkt to /Users/omad/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/omad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
timestr = time.strftime("%Y%m%d-%H%M%S")

# Saving processed files
X_train_filename_csv = f'../custom_data/processed/train_{timestr}.csv'
X_test_filename_csv = f'../custom_data/processed/test_{timestr}.csv'
X_validation_filename_csv = f'../custom_data/processed/validation_{timestr}.csv'

# Extract Features to csv file


### Training


In [8]:
values = load_parquet_values("../custom_data/raw/train_data.parquet")

extract_features_to_csv(X_train_filename_csv, values)

values = None

Starting ../custom_data/processed/train_20250512-133035.csv at 2025-05-12 13:30:35.079989. Rows=195, using 10 CPU cores
Exporting 1588 column features
Finished. Processed 195 rows in 0:00:01.688444, key_count=2


### Validation

In [9]:
values = load_parquet_values("../custom_data/raw/validation_data.parquet")

extract_features_to_csv(X_validation_filename_csv, values)

values = None

Starting ../custom_data/processed/validation_20250512-133035.csv at 2025-05-12 13:30:36.779150. Rows=24, using 10 CPU cores
Exporting 1588 column features
Finished. Processed 24 rows in 0:00:00.893086, key_count=1


### Test

In [10]:
values = load_parquet_values("../custom_data/raw/test_data.parquet")

extract_features_to_csv(X_test_filename_csv, values)

values = None

Starting ../custom_data/processed/test_20250512-133035.csv at 2025-05-12 13:30:37.683016. Rows=25, using 10 CPU cores
Exporting 1588 column features
Finished. Processed 25 rows in 0:00:00.459384, key_count=1


## Read locally processed features

### Train

In [11]:
start = datetime.now()

X_train = pd.read_csv(X_train_filename_csv, dtype=np.float32)

#print(f'Load Features (test) process took {datetime.now() - start} seconds.')

In [12]:
X_train.head()

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,0.000514,-0.001003,0.000674,0.000821,0.001041,-0.000385,-0.000496,-0.000719,-0.001180,0.000267
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,0.000544,0.000241,0.000429,-0.000016,0.001145,-0.000211,0.001034,0.000125,0.000107,-0.000239
2,1.0,1.0,64.0,0.0,64.0,64.0,64.0,64.0,-3.0,0.0,...,-0.000522,-0.000405,0.000569,0.000536,0.000913,0.000370,-0.000235,-0.001172,-0.000095,0.001129
3,1.0,1.0,194.0,0.0,194.0,194.0,194.0,194.0,-3.0,0.0,...,-0.000391,-0.000177,-0.000697,-0.000771,0.000656,0.000983,0.000140,-0.001013,-0.000352,0.000620
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.051159,-0.016158,-0.048814,0.005991,-0.023463,-0.008061,0.005730,-0.010927,-0.035978,-0.000916


### Validation

In [13]:
start = datetime.now()

X_validation = pd.read_csv(X_validation_filename_csv, dtype=np.float32)

#print(f'Load Features (test) process took {datetime.now() - start} seconds.')

In [14]:
X_validation.head()

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.000967,0.000460,0.001104,-0.000465,0.000576,0.000005,-0.000959,-0.000052,-0.000964,-0.000773
1,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,-3.0,0.0,...,-0.000226,0.000983,-0.000850,0.000092,-0.000407,0.000519,0.000510,-0.000682,-0.001222,0.000587
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.000138,-0.000095,-0.000669,0.000323,0.000245,-0.000022,0.000170,-0.000878,0.000074,-0.001163
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.000659,0.000801,0.000650,0.000953,0.000639,-0.000044,-0.000441,-0.000652,-0.001091,-0.000161
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,0.000692,0.000392,-0.000850,0.001200,0.000917,0.000730,0.000322,-0.000027,-0.000964,-0.000302


### Test

In [15]:
start = datetime.now()

X_test = pd.read_csv(X_test_filename_csv, dtype=np.float32)

#print(f'Load Features (test) process took {datetime.now() - start} seconds.')

In [16]:
X_test.head()

,n_[0]-agg-any,n_[0]-agg-all,n_[0]-agg-mean,n_[0]-agg-var,n_[0]-agg-min,n_[0]-agg-max,n_[0]-agg-median,n_[0]-agg-sum,n_[0]-agg-kurtosis,n_[0]-agg-skewness,...,par_vec_390,par_vec_391,par_vec_392,par_vec_393,par_vec_394,par_vec_395,par_vec_396,par_vec_397,par_vec_398,par_vec_399
0,1.0,1.0,237.0,0.0,237.0,237.0,237.0,237.0,-3.0,0.0,...,-0.001138,-0.001117,0.000982,-0.000464,0.001045,-0.000439,-0.001195,0.000171,-0.001174,0.000763
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.000999,0.001071,0.000260,-0.000683,0.001152,-0.000543,-0.001222,-0.000603,0.000798,0.000779
2,1.0,1.0,218.0,0.0,218.0,218.0,218.0,218.0,-3.0,0.0,...,0.000768,0.000368,0.000136,-0.000945,-0.001154,-0.001250,-0.000049,0.000453,0.000749,-0.000039
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-3.0,0.0,...,-0.000999,0.001071,0.000260,-0.000683,0.001152,-0.000543,-0.001222,-0.000603,0.000798,0.000779
4,1.0,1.0,915.0,0.0,915.0,915.0,915.0,915.0,-3.0,0.0,...,0.000300,-0.000656,0.000227,-0.001039,-0.001057,0.000508,0.000627,-0.001008,-0.000305,-0.000245


## Impute NaN values with feature means

In [17]:
start = datetime.now()

train_columns_means = pd.DataFrame(X_train.mean()).transpose()

#print(f'Transpose process took {datetime.now() - start} seconds.')

In [18]:
start = datetime.now()

X_train.fillna(train_columns_means.iloc[0], inplace=True)
X_validation.fillna(train_columns_means.iloc[0], inplace=True)
X_test.fillna(train_columns_means.iloc[0], inplace=True)

train_columns_means=None

#print(f'FillNA process took {datetime.now() - start} seconds.')

In [20]:
start = datetime.now()

X_train.to_parquet('../custom_data/processed/train.parquet', engine='pyarrow', compression='snappy')
X_validation.to_parquet('../custom_data/processed/validation.parquet', engine='pyarrow', compression='snappy')
X_test.to_parquet('../custom_data/processed/test.parquet', engine='pyarrow', compression='snappy')

#print(f'Save parquet process took {datetime.now() - start} seconds.')

In [21]:
print(f'Completed at {datetime.now()}.')

Completed at 2025-05-12 13:34:39.566356.
